In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

# Most reliable approach - resolves relative to the notebook file itself
# Walk up from cwd until we find the project root (identified by a known file)
project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.pipeline.props_pipeline.ppm_pipeline import *
from src.pipeline.props_pipeline.apm_pipeline import *
from src.pipeline.props_pipeline.rpm_pipeline import *
from src.pipeline.props_pipeline.min_pipeline import *
from src.live import *
from src.historical_analysis.dataScraper import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### Get updated lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
{'PHX': ['Haywood Highsmith', 'Dillon Brooks'], 'CHI': ['Matas Buzelis', 'Nick Richards', 'Josh Giddey'], 'MEM': ['Cam Spencer', 'Jahmai Mashack', 'Javon Small', 'Ty Jerome', 'Cedric Coward', 'GG Jackson'], 'MIL': ['Gary Trent', 'Bobby Portis'], 'WAS': ['Justin Champagnie', 'Tristan Vukcevic', 'Tre Johnson', 'Bilal Coulibaly', 'Alex Sarr'], 'BKN': ['Ben Saraf', 'Nic Claxton', 'Noah Clowney', 'Terance Mann', 'Ziaire Williams'], 'IND': ['Ben Sheppard', 'Jarace Walker'], 'MIN': ['Anthony Edwards'], 'DAL': ['Daniel Gafford']}

Out Players:
{'TOR': ['Chucky Hepburn'], 'MIL': ['Kevin Porter'], 'WAS': ["D'Angelo Russell", 'Anthony Davis'], 'IND': ['Andrew Nembhard', 'T.J. McConnell', 'Aaron Nesmith', 'Pascal Siakam'], 'CLE': ['Dean Wade', 'Jaylon Tyson', 'Sam Merrill', 'Evan Mobley', 'Jarrett Allen'], 'ORL': ['Anthony Black', 'Jett Howard', 'Jonathan Isaac'], 'NOP': ['Bryce McGowens', 'Karlo Matkovic', 'Dejounte Murray', 'Trey Murphy'], 'CHA': ['PJ Hall'], 'MIN': ['Jade

### Dataset

In [3]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
base_df = pd.concat([s25, s26])
base_df.tail()

,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,START_POSITION,pos,age
130,NaN,2025-26,1631172,Ousmane Dieng,Ousmane,1610612749,MIL,Milwaukee Bucks,22501126,2026-04-03T00:00:00,MIL vs. BOS,L,23.983333,4,15,0.267,1,4,0.250,0,0,0.0,0,3,3,1,1,0,0,1,2,0,9,-32,13.1,0,0,14.0,1,23:59,1,94.0,97.9,97.9,155.5,159.2,159.2,-61.5,-61.3,-61.3,0.091,1.0,5.9,0.000,0.158,0.060,5.9,5.9,0.300,0.300,0.296,0.297,99.19,96.07,80.06,96.07,-0.009,47,4.0,15.0,35,83,0.422,21,47,0.447,10,13,0.769,8,22,30,24,13.0,5,4,4,21,13,101,-32.0,107.8,109.8,138.0,141.5,-30.3,-31.7,0.686,1.85,19.0,0.200,0.600,0.378,0.141,0.548,0.569,95.0,93.0,77.50,92,0.322,1610612738,BOS,Boston Celtics,50,89,0.562,17,37,0.459,16,19,0.842,10,38,48,33,9.0,7,4,4,13,21,133,32.0,138.0,141.5,107.8,109.8,30.3,31.7,0.660,3.67,23.1,0.400,0.800,0.622,0.096,0.657,0.683,95.0,93.0,77.50,94,0.678,G,C,22.0
129,NaN,2025-26,1629013,Landry Shamet,Landry,1610612752,NYK,New York Knicks,22501123,2026-04-03T00:00:00,NYK vs. CHI,W,12.683333,3,7,0.429,2,4,0.500,0,0,0.0,1,0,1,1,0,1,0,0,0,0,8,12,13.7,0,0,14.0,1,12:41,1,143.6,137.0,137.0,95.9,92.6,92.6,47.8,44.4,44.4,0.083,0.0,12.5,0.083,0.000,0.037,0.0,0.0,0.571,0.571,0.233,0.243,98.09,102.18,85.15,102.18,0.105,27,3.0,7.0,48,91,0.527,15,39,0.385,25,28,0.893,13,41,54,30,9.0,11,3,2,21,23,136,40.0,136.9,136.0,96.5,96.0,40.4,40.0,0.625,3.33,20.7,0.333,0.788,0.577,0.090,0.610,0.658,99.4,100.0,83.33,100,0.713,1610612741,CHI,Chicago Bulls,35,81,0.432,11,35,0.314,15,26,0.577,9,27,36,24,16.0,4,2,3,23,21,96,-40.0,96.5,96.0,136.9,136.0,-40.4,-40.0,0.686,1.50,17.8,0.212,0.667,0.423,0.160,0.500,0.519,99.4,100.0,83.33,100,0.287,NaN,SG,28.0
128,NaN,2025-26,1630182,Josh Green,Josh,1610612766,CHA,Charlotte Hornets,22501120,2026-04-03T00:00:00,CHA vs. IND,W,24.400000,2,5,0.400,2,4,0.500,0,0,0.0,1,1,2,2,0,1,0,0,2,0,6,8,14.4,0,0,14.0,1,24:24,1,124.2,125.5,125.5,109.4,107.7,107.7,14.8,17.8,17.8,0.100,0.0,28.6,0.036,0.036,0.036,0.0,0.0,0.600,0.600,0.088,0.088,101.04,101.31,84.43,101.31,0.044,51,2.0,5.0,46,96,0.479,24,49,0.490,13,15,0.867,12,36,48,31,10.0,8,7,1,18,18,129,21.0,128.2,130.3,109.8,109.1,18.5,21.2,0.674,3.10,21.5,0.275,0.722,0.505,0.101,0.604,0.629,99.5,99.0,82.50,99,0.574,1610612754,IND,Indiana Pacers,43,93,0.462,15,36,0.417,7,10,0.700,11,34,45,29,12.0,5,1,7,18,18,108,-21.0,109.8,109.1,128.2,130.3,-18.5,-21.2,0.674,2.42,20.6,0.278,0.725,0.495,0.121,0.543,0.554,99.5,99.0,82.50,99,0.426,NaN,SG,25.0
139,NaN,2025-26,203937,Kyle Anderson,Kyle,1610612750,MIN

### Load latest odds on file

In [4]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_dds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_dds = pd.json_normalize(data)

print("Loaded:", file.name)
team_dds.head()

Loaded: NBA_20260405_122344.json


,home_team,away_team,commence_time,bookmakers
0,Boston Celtics,Toronto Raptors,2026-04-05 19:40:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
1,Brooklyn Nets,Washington Wizards,2026-04-05 19:40:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
2,Chicago Bulls,Phoenix Suns,2026-04-05 19:40:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
3,Milwaukee Bucks,Memphis Grizzlies,2026-04-05 19:40:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
4,Cleveland Cavaliers,Indiana Pacers,2026-04-05 22:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."


In [5]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

#load season stats
pts_df = pd.read_csv('data/processed/training/S26_TRAINING_PPM.csv')
ast_df = pd.read_csv('data/processed/training/S26_TRAINING_APM.csv')
reb_df = pd.read_csv('data/processed/training/S26_TRAINING_RPM.csv')
min_df = pd.read_csv('data/processed/training/S26_TRAINING_MIN.csv')

#load dfs lines
lines_dfs = pd.read_csv(dfs_file)
lines_dfs_pts = lines_dfs[(lines_dfs['BOOKMAKER'] == 'Underdog') & (lines_dfs['CATEGORY'] == 'player_points')]
lines_dfs_ast = lines_dfs[(lines_dfs['BOOKMAKER'] == 'Underdog') & (lines_dfs['CATEGORY'] == 'player_assists')]
lines_dfs_reb = lines_dfs[(lines_dfs['BOOKMAKER'] == 'Underdog') & (lines_dfs['CATEGORY'] == 'player_rebounds')]
pts_names = lines_dfs_pts['NAME'].unique()
ast_names = lines_dfs_ast['NAME'].unique()
reb_names = lines_dfs_reb['NAME'].unique()

#load us lines with actual odds
lines_us = pd.read_csv(us_file)
lines_us_pts = lines_us[(lines_us['CATEGORY'] == 'player_points')]
lines_us_ast = lines_us[(lines_us['CATEGORY'] == 'player_assists')]
lines_us_reb = lines_us[(lines_us['CATEGORY'] == 'player_rebounds')]

print(f"DFS latest pull: {lines_dfs['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {lines_us['DATA_PULLED_AT'].max()}")
lines_us_pts.head()

DFS latest pull: 2026-04-05 12:21:00
US latest pull: 2026-04-05 12:23:44


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,FanDuel,player_points,Jakob Poeltl,Over,10.5,-102,2026-04-05,2026-04-05T19:23:43Z,2026-04-05 12:23:44
1,FanDuel,player_points,Jakob Poeltl,Under,10.5,-125,2026-04-05,2026-04-05T19:23:43Z,2026-04-05 12:23:44
2,FanDuel,player_points,Ja'Kobe Walter,Over,9.5,104,2026-04-05,2026-04-05T19:23:43Z,2026-04-05 12:23:44
3,FanDuel,player_points,Ja'Kobe Walter,Under,9.5,-132,2026-04-05,2026-04-05T19:23:43Z,2026-04-05 12:23:44
4,FanDuel,player_points,Payton Pritchard,Over,15.5,-113,2026-04-05,2026-04-05T19:23:43Z,2026-04-05 12:23:44


### Load my models

In [6]:
import joblib

#minutes
min_bundle = joblib.load("src/models/saved_models/min_quantile_xgb.joblib")
min_quantile_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]

#points per minute
ppm_bundle = joblib.load("src/models/saved_models/ppm_quantile_xgb.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]

#assists per minute
apm_bundle = joblib.load("src/models/saved_models/apm_quantile_xgb.joblib")
apm_quantile_models = apm_bundle["quantile_models"]
apm_feature_names = apm_bundle["feature_names"]

#rebounds per minute
rpm_bundle = joblib.load("src/models/saved_models/rpm_quantile_xgb.joblib")
rpm_quantile_models = rpm_bundle["quantile_models"]
rpm_feature_names = rpm_bundle["feature_names"]

### Get Min predictions and Stat Per Min predictions 

In [7]:
pts_preds = predict_min_times_rate(
    pts_names, min_df, pts_df, current_date,
    name_dict=nameDict,
    rate_pipeline=ppm_pipeline,
    rate_quantile_models=ppm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="PTS",
)
ast_preds = predict_min_times_rate(
    ast_names, min_df, ast_df, current_date,
    name_dict=nameDict,
    rate_pipeline=apm_pipeline,
    rate_quantile_models=apm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="AST",
)
reb_preds = predict_min_times_rate(
    reb_names, min_df, reb_df, current_date,
    name_dict=nameDict,
    rate_pipeline=rpm_pipeline,
    rate_quantile_models=rpm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="REB",
)
ast_preds.head(10)

[SKIP] R.J. Barrett: single positional indexer is out-of-bounds
[SKIP] Anthony Gill: single positional indexer is out-of-bounds
[SKIP] E.J. Liddell: single positional indexer is out-of-bounds
[SKIP] Leonard Miller: single positional indexer is out-of-bounds
[SKIP] Kobe Brown: single positional indexer is out-of-bounds
[SKIP] Wendell Carter Jr: single positional indexer is out-of-bounds
[SKIP] Herb Jones: single positional indexer is out-of-bounds
[SKIP] Devin Carter: single positional indexer is out-of-bounds
[SKIP] Devin Carter: single positional indexer is out-of-bounds
[SKIP] E.J. Liddell: single positional indexer is out-of-bounds
[SKIP] A.J. Green: single positional indexer is out-of-bounds


,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,STAT_Q10,STAT_Q50,STAT_Q90
0,Scottie Barnes,AST,26.77,30.67,38.59,0.1082,0.2375,0.3317,2.90,7.28,12.80
1,Bub Carrington,AST,19.80,27.83,35.14,0.1015,0.1702,0.2831,2.01,4.74,9.95
2,Leaky Black,AST,25.79,29.62,37.82,0.0225,0.0531,0.1315,0.58,1.57,4.97
3,Drake Powell,AST,15.87,25.51,32.99,-0.0004,0.0568,0.1368,-0.01,1.45,4.51
4,Devin Booker,AST,27.54,33.95,39.88,0.0954,0.1846,0.2959,2.63,6.27,11.80
5,Walter Clayton Jr.,AST,14.86,23.50,31.02,0.0819,0.1895,0.3175,1.22,4.45,9.85
6,Mike Conley,AST,15.20,24.53,33.99,0.1111,0.2057,0.3106,1.69,5.05,10.56
7,Cooper Flagg,AST,27.99,33.88,38.41,0.0520,0.1306,0.2248,1.45,4.43,8.64
8,Naji Marshall,AST,19.47,26.38,34.16,0.0306,0.1028,0.2042,0.60,2.71,6.97
9,Kawhi Leonard,AST,26.28,31.96,38.44,0.0350,0.1123,0.2006,0.92,3.59,7.71


### Get Line Probabilities

In [8]:
all_line_probs = pd.concat([
    line_probs_for_market(ast_preds, lines_dfs_ast, nameDict, run_stat_simulation),
    line_probs_for_market(reb_preds, lines_dfs_reb, nameDict, run_stat_simulation),
    line_probs_for_market(pts_preds, lines_dfs_pts, nameDict, run_pts_simulation),
], ignore_index=True)
all_line_probs.sample(10)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER
3,Drake Powell,AST,2.5,25.51,1.45,0.337,0.663
34,Tari Eason,REB,5.5,23.51,6.67,0.684,0.316
35,Jayson Tatum,PTS,22.5,35.40,26.63,0.713,0.287
108,Draymond Green,PTS,8.5,27.63,7.15,0.563,0.437
72,Moussa Diabaté,PTS,7.5,26.20,7.27,0.642,0.357
88,Ajay Mitchell,PTS,11.5,23.01,11.80,0.619,0.381
101,John Collins,PTS,12.5,26.09,14.22,0.678,0.322
100,Brook Lopez,PTS,10.5,26.65,11.50,0.712,0.288
37,Jakob Poeltl,PTS,9.5,25.38,11.33,0.783,0.217
62,Quenton Jackson,PTS,13.5,22.44,10.76,0.357,0.643


In [9]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Underdog',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

all_line_probs = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
all_line_probs.head(10)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER,CATEGORY,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
0,Scottie Barnes,AST,6.5,30.67,7.28,0.621,0.379,AST,Boston Celtics,9.0,220.0,111.7,4.0,95.49,30.0,-103.0,110.0,0.507,0.476,9.4,10.0,3.63,2.9,3.5,-0.799,0.788,0.212,55.30,-55.48,0.8,0.7,0.53,0.34,29.80,5.16,0.22,0.03,6.20,5.0
3,Drake Powell,AST,2.5,25.51,1.45,0.337,0.663,AST,Washington Wizards,-3.0,231.0,121.3,30.0,102.38,6.0,-119.0,-109.0,0.543,0.522,1.6,1.5,1.43,-0.9,-1.0,0.629,0.265,0.735,-51.23,40.93,0.2,0.3,0.20,0.22,26.03,5.69,0.14,0.05,1.67,3.0
4,Devin Booker,AST,6.5,33.95,6.27,0.546,0.454,AST,Chicago Bulls,-11.5,242.0,117.8,23.0,102.94,3.0,104.0,-107.0,0.490,0.517,6.1,6.0,1.52,-0.4,-0.5,0.263,0.396,0.604,-19.22,16.85,0.6,0.4,0.40,0.49,33.22,4.00,0.31,0.06,5.67,3.0
7,Cooper Flagg,AST,5.5,33.88,4.43,0.410,0.590,AST,Los Angeles Lakers,1.5,233.5,115.7,20.0,99.34,21.0,-104.0,-106.0,0.510,0.515,5.6,6.0,3.20,0.1,0.5,-0.031,0.512,0.488,0.43,-5.16,0.2,0.5,0.53,0.37,34.77,3.99,0.30,0.05,8.50,2.0
8,Naji Marshall,AST,3.5,26.38,2.71,0.463,0.537,AST,Los Angeles Lakers,1.5,233.5,115.7,20.0,99.34,21.0,-105.0,-110.0,0.512,0.524,4.3,3.5,1.95,0.8,0.0,-0.410,0.659,0.341,28.66,-34.90,0.4,0.5,0.47,0.35,30.22,4.68,0.25,0.05,3.50,6.0
10,Nique Clifford,AST,3.5,29.82,2.99,0.504,0.496,AST,Los Angeles Clippers,13.5,228.5,115.2,19.0,97.22,28.0,-108.0,100.0,0.519,0.500,4.3,4.0,2.16,0.8,0.5,-0.370,0.644,0.356,24.03,-28.80,0.6,0.6,0.60,0.27,33.54,6.10,0.18,0.04,2.67,3.0
11,Stephen Curry,AST,3.5,29.81,5.61,0.763,0.237,AST,Houston Rockets,3.5,226.5,112.1,6.0,96.83,29.0,105.0,-114.0,0.488,0.533,5.4,5.0,3.60,1.9,1.5,-0.528,0.701,0.299,43.70,-43.87,0.2,0.6,0.73,0.78,28.85,3.68,0.29,0.06,5.25,4.0
12,Brandin Podziemski,AST,3.5,28.47,4.06,0.608,0.392,AST,Houston Rockets,3.5,226.5,112.1,6.0,96.83,29.0,-120.0,104.0,0.545,0.490,4.1,4.5,1.85,0.6,1.0,-0.324,0.627,0.373,14.95,-23.91,0.8,0.6,0.67,0.50,30.39,6.48,0.21,0.04,3.43,7.0
13,Scottie Barnes,REB,6.5,30.67,6.31,0.594,0.406,REB,Boston Celtics,9.0,220.0,111.7,4.0,95.49,30.0,-105.0,-106.0,0.512,0.515,5.5,5.5,2.22,-1.0,-1.0,0.450,0.326,0.674,-36.35,30.98,0.0,0.4,0.33,0.59,29.80,5.16,0.22,0.03,7.80,5.0
14,Nolan Traore,REB,2.5,26.17,2.01,0.417,0.583,REB,Washington Wizards,-3.0,231.0,121.3,30.0,102.38,6.0,-125.0,104.0,0.556,0.490,2.3,2.0,1.25,-0.2,-0.5,0.160,0.436,0.564,-21.52,15.06,0.6,0.4,0.33,0.26,23.42,5.23,0.29,0.04,2.00,2.0


### Get top EVs

In [ ]:
slate_path = build_greedy_slate(
    prob_df=all_line_probs,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=20,
    json_path="data/props/ev_analysis/greedy_slate.json",
)
print(slate_path)

Legs: 86  |  Pairs: 257  |  Slate: 7  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/greedy_slate.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/greedy_slate.json
